# NumPy 2 — indexing, broadcasting, vectorisation

Every section starts with a tiny array. Real data appears in one short cell per section.

**What's in here**
- basic, fancy and boolean indexing
- `np.where`, `np.select`, `np.clip`
- broadcasting, with shapes printed
- the `(n,)` vs `(n,1)` trap
- `axis` semantics
- vectorising a loop
- `cumsum`, `diff`, `searchsorted`, `argsort`, `argmax`, `unique`
- comparing floats with `isclose`

In [1]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.width", 120)

df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
cons = df["consumption_mwh"].to_numpy()
temp = df["temp_c"].to_numpy()
price = df["price_eur_mwh"].to_numpy()
hour = df["time"].dt.hour.to_numpy()

## 1. Basic indexing and slicing

Positions start at 0. Negative positions count from the end.

In [2]:
a = np.array([10, 20, 30, 40, 50])
print(a[0])
print(a[-1])

10
50


`a[start:stop]` excludes the stop. `a[::step]` takes every step-th element.

In [3]:
print(a[1:3])
print(a[::2])
print(a[::-1])       # reversed

[20 30]
[10 30 50]
[50 40 30 20 10]


2-D: one bracket, `m[row, col]`. A colon means "all".

In [4]:
m = np.array([[0, 1, 2],
              [3, 4, 5]])
print(m[1, 2])       # row 1, column 2
print(m[1])          # whole row 1
print(m[:, 2])       # whole column 2

5
[3 4 5]
[2 5]


## 2. Fancy indexing

A list of positions picks those elements, in that order (repeats allowed). The result is a copy.

In [5]:
a = np.array([10, 20, 30, 40, 50])
a[[0, 3, 3, -1]]

array([10, 40, 40, 50])

**Pitfall:** on a 2-D array, `m[[0, 1], [1, 2]]` pairs the rows with the columns
elementwise: you get `m[0,1]` and `m[1,2]`, not a block.

In [6]:
m = np.array([[0, 1, 2],
              [3, 4, 5]])
m[[0, 1], [1, 2]]

array([1, 5])

For the block use `np.ix_`.

In [7]:
m[np.ix_([0, 1], [1, 2])]

array([[1, 2],
       [4, 5]])

## 3. Boolean indexing

A comparison gives an array of True/False.

In [8]:
a = np.array([10, 20, 30, 40, 50])
mask = a > 25
mask

array([False, False,  True,  True,  True])

Indexing with the mask keeps the True positions.

In [9]:
a[mask]

array([30, 40, 50])

`mask.sum()` counts the Trues; `mask.mean()` is the fraction.

In [10]:
print(mask.sum())
print(mask.mean())

3
0.6


Combine conditions with `&`, `|`, `~`, each condition in parentheses.

In [11]:
a[(a > 15) & (a < 45)]

array([20, 30, 40])

**Pitfall:** Python's `and` does not work on arrays.

In [12]:
try:
    (a > 15) and (a < 45)
except ValueError as e:
    print("ValueError:", e)

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()


Assigning through a mask changes the array in place. This is how you clean values.

In [13]:
p = np.array([50., -5., 80., -2.])
p[p < 0] = 0.0
p

array([50.,  0., 80.,  0.])

Real data: how many cold hours, and mean consumption in them.

In [14]:
cold = temp < 0
print(cold.sum(), "cold hours")
print(cons[cold].mean().round(0), "MWh mean when cold")
print(cons[~cold].mean().round(0), "MWh mean otherwise")

1139 cold hours
30058.0 MWh mean when cold
29264.0 MWh mean otherwise


## 4. `np.where`, `np.select`, `np.clip`

`np.where(condition, a, b)` is a vectorised if/else: `a` where True, `b` where False.

In [15]:
x = np.array([1, 5, 10])
np.where(x > 4, "big", "small")

array(['small', 'big', 'big'], dtype='<U5')

With one argument it returns the **positions** where the condition is True.

In [16]:
np.where(x > 4)

(array([1, 2]),)

`np.select` is if / elif / else: the first matching condition wins.

In [17]:
t = np.array([2, 10, 18, 25])
np.select([t < 5, t < 15, t < 22], ["cold", "mild", "warm"], default="hot")

array(['cold', 'mild', 'warm', 'hot'], dtype='<U4')

`np.clip` caps values at a floor and a ceiling.

In [18]:
np.clip(np.array([-5, 50, 500]), 0, 200)

array([  0,  50, 200])

Real data: label peak hours.

In [19]:
peak = np.where((hour >= 16) & (hour <= 19), "peak", "offpeak")
print(peak[14:22])

['offpeak' 'offpeak' 'peak' 'peak' 'peak' 'peak' 'offpeak' 'offpeak']


## 5. Broadcasting

A scalar is applied to every element.

In [20]:
m = np.array([[0, 1, 2],
              [3, 4, 5]])
m * 10

array([[ 0, 10, 20],
       [30, 40, 50]])

A `(3,)` row vector against a `(2, 3)` matrix: the row is applied to **each row**.

In [21]:
row = np.array([100, 200, 300])
print(m.shape, row.shape)
m + row

(2, 3) (3,)


array([[100, 201, 302],
       [103, 204, 305]])

A `(2, 1)` column vector: the column is applied to **each column**.

In [22]:
col = np.array([[100],
                [200]])
print(m.shape, col.shape)
m + col

(2, 3) (2, 1)


array([[100, 101, 102],
       [203, 204, 205]])

The rule: align shapes from the right; each pair must be equal or one of them 1.
`(2, 3)` with `(2,)` fails because 3 ≠ 2.

In [23]:
try:
    m + np.array([100, 200])
except ValueError as e:
    print("ValueError:", e)

ValueError: operands could not be broadcast together with shapes (2,3) (2,) 


Useful pattern: subtract the column means (a `(3,)` vector) from every row.

In [24]:
means = m.mean(axis=0)
print(means)
m - means

[1.5 2.5 3.5]


array([[-1.5, -1.5, -1.5],
       [ 1.5,  1.5,  1.5]])

Outer operation without a loop: `a[:, None] - a[None, :]` gives every pairwise difference.

In [25]:
a = np.array([1, 2, 4])
print(a[:, None].shape, a[None, :].shape)
a[:, None] - a[None, :]

(3, 1) (1, 3)


array([[ 0, -1, -3],
       [ 1,  0, -2],
       [ 3,  2,  0]])

Real data: z-score three columns at once.

In [26]:
X = np.column_stack([temp, cons, price])
Z = (X - X.mean(axis=0)) / X.std(axis=0)
print(Z.mean(axis=0).round(6))
print(Z.std(axis=0).round(6))

[-0.  0.  0.]
[1. 1. 1.]


### The `(n,)` vs `(n,1)` trap

`y` is `(3,)`, `pred` is `(3,)`: the difference is `(3,)`. Good.

In [27]:
y = np.array([1., 2., 3.])
pred = np.array([1.1, 1.9, 3.2])
print((y - pred).shape)
print(y - pred)

(3,)
[-0.1  0.1 -0.2]


Now make `y` a column `(3, 1)`. Broadcasting turns the subtraction into a 3×3 table
of every y against every pred. No error, wrong answer.

In [28]:
y_col = y.reshape(-1, 1)
print((y_col - pred).shape)
print(y_col - pred)

(3, 3)
[[-0.1 -0.9 -2.2]
 [ 0.9  0.1 -1.2]
 [ 1.9  1.1 -0.2]]


Fix: `ravel()` the column back to 1-D.

In [29]:
print((y_col.ravel() - pred).shape)

(3,)


**Interview check:** "The residuals have shape (17520, 17520)." → one side was a column
(from `df[["y"]].values` or `reshape(-1, 1)`), the other 1-D. With 17,520 rows that is 2.5 GB.

## 6. `axis` semantics

`axis` is the dimension that gets **collapsed**.

In [30]:
m = np.array([[0, 1, 2],
              [3, 4, 5]])
m

array([[0, 1, 2],
       [3, 4, 5]])

In [31]:
m.sum()              # everything

15

In [32]:
m.sum(axis=0)        # collapse rows -> one value per column: 0+3, 1+4, 2+5

array([3, 5, 7])

In [33]:
m.sum(axis=1)        # collapse columns -> one value per row: 0+1+2, 3+4+5

array([ 3, 12])

`keepdims=True` keeps the collapsed axis with size 1, so the result broadcasts back.

In [34]:
print(m.mean(axis=1).shape)
print(m.mean(axis=1, keepdims=True).shape)
m - m.mean(axis=1, keepdims=True)

(2,)
(2, 1)


array([[-1.,  0.,  1.],
       [-1.,  0.,  1.]])

Real data: reshape into (days, 24) and average over days (axis=0) to get the hourly profile.

In [35]:
by_day = cons.reshape(-1, 24)
print(by_day.shape)
profile = by_day.mean(axis=0)
print(profile[:6].round(0))

(730, 24)
[25430. 24372. 23875. 23610. 23883. 24989.]


## 7. Vectorising a loop

Heating degrees: `max(15 - temp, 0)`. First the loop on 5 values.

In [36]:
t = np.array([5., 10., 15., 20., 25.])
out = np.empty(5)
for i in range(5):
    out[i] = max(15 - t[i], 0)
out

array([10.,  5.,  0.,  0.,  0.])

The same in one array expression.

In [37]:
np.maximum(15 - t, 0)

array([10.,  5.,  0.,  0.,  0.])

On 17,520 rows the vectorised form is much faster.

In [38]:
def heating_loop(t):
    out = np.empty(len(t))
    for i in range(len(t)):
        out[i] = max(15 - t[i], 0)
    return out

print(np.allclose(heating_loop(temp), np.maximum(15 - temp, 0)))

True


In [39]:
%timeit heating_loop(temp)
%timeit np.maximum(15 - temp, 0)

4.72 ms ± 278 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


15.2 µs ± 578 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


Some loops cannot be vectorised: each step depends on the previous result.
An EWMA `y[t] = a*x[t] + (1-a)*y[t-1]` on 4 values:

In [40]:
x = np.array([10., 20., 30., 40.])
a = 0.5
y = np.empty(4)
y[0] = x[0]
for t in range(1, 4):
    y[t] = a * x[t] + (1 - a) * y[t - 1]
print(y)
print(pd.Series(x).ewm(alpha=a, adjust=False).mean().to_numpy())   # pandas does the loop for you

[10.   15.   22.5  31.25]
[10.   15.   22.5  31.25]


## 8. Cumulative operations and differences

In [41]:
x = np.array([1., 2., 3., 4.])
print(np.cumsum(x))
print(np.cumprod(x))

[ 1.  3.  6. 10.]
[ 1.  2.  6. 24.]


`np.diff` is `x[1:] - x[:-1]`: one element **shorter**. `prepend=np.nan` keeps the length.

In [42]:
print(np.diff(x))
print(np.diff(x, prepend=np.nan))

[1. 1. 1.]
[nan  1.  1.  1.]


Compounding returns: cumulative product of `1 + r`.

In [43]:
r = np.array([0.10, -0.10, 0.10])
np.cumprod(1 + r)

array([1.1  , 0.99 , 1.089])

## 9. `searchsorted` — binning

Given a **sorted** array of edges, `searchsorted` says where a value would be inserted.

In [44]:
edges = np.array([0, 10, 20])
np.searchsorted(edges, [5, 10, 15, 25])

array([1, 1, 2, 3])

5 goes in position 1 (between 0 and 10), 15 in position 2, 25 at the end (3).
10 is a tie: `side="left"` (default) puts it before the 10, `side="right"` after.

In [45]:
np.searchsorted(edges, [5, 10, 15, 25], side="right")

array([1, 2, 2, 3])

**Pitfall:** the edges must be sorted; NumPy does not check.

Real data: temperature bands and the mean consumption in each.

In [46]:
edges = np.array([0, 10, 20])
band = np.searchsorted(edges, temp, side="right")
for b in [0, 1, 2, 3]:
    print("band", b, "->", cons[band == b].mean().round(0), "MWh")

band 0 -> 30058.0 MWh
band 1 -> 30652.0 MWh
band 2 -> 27797.0 MWh
band 3 -> 29797.0 MWh


## 10. `argsort`, `argmax`, `argmin`

`argsort` returns the **positions** that would sort the array.

In [47]:
a = np.array([30, 10, 20])
order = np.argsort(a)
print(order)
print(a[order])          # sorted values

[1 2 0]
[10 20 30]


`argmax` is the position of the (first) maximum.

In [48]:
print(a.argmax())
print(a[a.argmax()])

0
30


Top-3 positions: last 3 of the ascending order, reversed.

In [49]:
top3 = np.argsort(price)[-3:][::-1]
print(top3)
df.loc[top3, ["time", "price_eur_mwh"]]

[6691 6237 7834]


,time,price_eur_mwh
6691,2022-10-06 19:00:00+00:00,419.60
6237,2022-09-17 21:00:00+00:00,379.06
7834,2022-11-23 10:00:00+00:00,375.29


## 11. `np.unique`

Sorted distinct values. `return_counts=True` gives a frequency table.

In [50]:
x = np.array(["b", "a", "b", "c", "b"])
vals, counts = np.unique(x, return_counts=True)
print(vals)
print(counts)

['a' 'b' 'c']
[1 3 1]


`return_inverse=True` gives integer codes (a simple label encoder).

In [51]:
vals, codes = np.unique(x, return_inverse=True)
print(codes)
print(vals[codes])       # decode back

[1 0 1 2 1]
['b' 'a' 'b' 'c' 'b']


## 12. Comparing floats

`0.1 + 0.2` is not exactly `0.3` in floating point.

In [52]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)
print(np.isclose(0.1 + 0.2, 0.3))

0.30000000000000004
False
True


`np.allclose` checks a whole array with a tolerance.

In [53]:
x = cons / 3 * 3
print((x != cons).sum(), "elements differ exactly")
print(np.allclose(x, cons))

2954 elements differ exactly
True


**Pitfall:** `isclose` says NaN ≠ NaN unless you pass `equal_nan=True`.

In [54]:
print(np.isclose(np.nan, np.nan))
print(np.isclose(np.nan, np.nan, equal_nan=True))

False
True


## Quick reference

| Task | Code |
|---|---|
| if/else vectorised | `np.where(cond, a, b)` |
| if/elif/else | `np.select([c1, c2], [v1, v2], default)` |
| cap values | `np.clip(x, lo, hi)` |
| per-column stat | `X.mean(axis=0)` |
| per-row stat | `X.mean(axis=1)` |
| column vector | `x[:, None]` |
| pairwise / outer | `a[:, None] - a[None, :]` |
| bin into edges | `np.searchsorted(edges, x, side="right")` |
| top-k positions | `np.argsort(x)[-k:][::-1]` |
| frequency table | `np.unique(x, return_counts=True)` |
| float equality | `np.allclose(a, b)` |